In [2]:
# Recharge automatiquement les modules du projet quand leur code change.
# ATTENTION : après BEAUCOUP de modifs de env/, un simple reload ne suffit pas
# (imports croisés -> constantes périmées). Le garde-fou en bas force alors un
# "Kernel > Restart Kernel".
%load_ext autoreload
%autoreload 2

# --- Phase 2 : Deep Q-Network "from scratch" (TensorFlow / Keras) -------------
# Même environnement, même récompense que la Q-table (notebook 07) : marge de
# contribution du bloc de décision, normalisée par épisode
#   = (prix - marginal_cost + ancillary_capture * FCI) * ventes / (1.5 * demande convertible)
# Observation 7 dims dont popularité + Fan Cost Index. Prix révisé tous les
# `steps_per_decision` pas de marché (cadence grossière -> horizon court).
# On remplace la table discrète par un réseau qui approxime Q(s, a) : gère l'état
# continu sans discrétisation et GÉNÉRALISE entre scénarios voisins.
#
# Prérequis : pip install -r ../requirements-drl.txt   (tensorflow)

import importlib
import os
import sys
import time

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")

import numpy as np
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath("../env"))
sys.path.append(os.path.abspath("../pricing_agent"))

import tensorflow as tf

# Recharge la chaîne complète DANS L'ORDRE des dépendances (noms en clair -> pas
# de collision avec une variable locale) :
#   refdata (constantes) -> agents (acheteur MNL) -> market (simulateur) -> gym_wrapper (env RL)
def _reload_project():
    for _name in ("refdata", "agents", "market", "gym_wrapper"):
        importlib.reload(importlib.import_module(_name))

_reload_project()
from gym_wrapper import TicketEnv
from dqn import ReplayBuffer, build_qnetwork, make_train_step

HORIZON = 120
env = TicketEnv(horizon=HORIZON)
obs_dim = int(env.observation_space.shape[0])
n_actions = int(env.action_space.n)

# Garde-fou : si l'env n'a pas les champs récents, le reload a échoué -> redémarrer le kernel.
_obs0, _info0 = env.reset(seed=0)
if obs_dim != _obs0.shape[0] or "ca_total" not in _info0 or not hasattr(env, "steps_per_decision"):
    raise RuntimeError("Environnement périmé — fais 'Kernel > Restart Kernel' puis Run All.")

print("TensorFlow", tf.__version__, "| obs_dim", obs_dim, "| actions", n_actions,
      "| décisions/épisode", HORIZON // env.steps_per_decision)

TensorFlow 2.21.0 | obs_dim 7 | actions 10 | décisions/épisode 12


In [3]:
# --- Hyperparamètres & composants DQN ---------------------------------------
# Cadence de décision de l'env (steps_per_decision) -> ~12 décisions / épisode,
# donc horizon court : le DQN apprend bien en TD standard (le problème n'était
# jamais l'algo mais l'horizon de 120).
DECISIONS_PER_EP = HORIZON // env.steps_per_decision

GAMMA = 0.97               # actualisation (identique au tabulaire)
LR = 5e-4                  # pas d'apprentissage Adam
BATCH_SIZE = 64
BUFFER_CAPACITY = 200_000
WARMUP_STEPS = 1_500       # remplir le buffer avant la 1re descente de gradient
TRAIN_EVERY = 1           # 1 descente de gradient par décision (il y en a peu)
TARGET_UPDATE_EVERY = 1_000
EPISODES = 15000           # ~1 h CPU
EPS_START, EPS_MIN, EPS_FRAC = 1.0, 0.06, 0.75   # ε linéaire, min atteint à 75 %

tf.keras.utils.set_random_seed(0)
rng = np.random.default_rng(0)

online = build_qnetwork(obs_dim, n_actions)          # MLP 128x128 (cf. pricing_agent/dqn.py)
target = build_qnetwork(obs_dim, n_actions)          # réseau cible (stabilise la cible)
target.set_weights(online.get_weights())
optimizer = tf.keras.optimizers.Adam(LR)
train_step = make_train_step(online, target, optimizer, GAMMA)
buffer = ReplayBuffer(BUFFER_CAPACITY, obs_dim)

online.summary()

total_steps_est = EPISODES * DECISIONS_PER_EP

def epsilon(step):
    if step >= EPS_FRAC * total_steps_est:
        return EPS_MIN
    return EPS_START + (EPS_MIN - EPS_START) * step / (EPS_FRAC * total_steps_est)

Model: "qnet"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,826 (73.54 KB)

 Trainable params: 18,826 (73.54 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# --- Boucle d'entraînement DQN ---------------------------------------------
# (pour un entraînement long avec checkpoint/reprise, préférer
#  `python ../scripts/train_dqn.py --episodes 2500`)

@tf.function(reduce_retracing=True)
def q_online(s):
    return online(s, training=False)

episode_returns = []
global_step = 0
t0 = time.time()

for ep in range(EPISODES):
    obs, _ = env.reset(seed=ep)                 # DR reproductible par épisode
    done = False
    ep_return = 0.0

    while not done:
        eps = epsilon(global_step)
        if rng.random() < eps:
            action = int(rng.integers(n_actions))                       # exploration
        else:
            action = int(tf.argmax(q_online(obs[None, :])[0]).numpy())  # exploitation

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        buffer.add(obs, action, reward, next_obs, terminated)           # bootstrap ssi terminé
        obs = next_obs
        ep_return += reward
        global_step += 1

        if buffer.size >= WARMUP_STEPS and global_step % TRAIN_EVERY == 0:
            batch = [tf.convert_to_tensor(x) for x in buffer.sample(BATCH_SIZE, rng)]
            train_step(*batch)
        if global_step % TARGET_UPDATE_EVERY == 0:
            target.set_weights(online.get_weights())                    # mise à jour du réseau cible

    episode_returns.append(ep_return)
    if (ep + 1) % 100 == 0:
        rate = (ep + 1) / (time.time() - t0)
        print(f"  épisode {ep + 1:>5d}/{EPISODES}  eps={epsilon(global_step):.3f}  "
              f"retour_moyen={np.mean(episode_returns[-100:]):.2f}  ({rate:.1f} ép/s)")

print("Entraînement terminé.")

  épisode   100/15000  eps=0.995  retour_moyen=31.94  (16.1 ép/s)
  épisode   200/15000  eps=0.989  retour_moyen=31.69  (14.1 ép/s)
  épisode   300/15000  eps=0.984  retour_moyen=33.17  (12.8 ép/s)
  épisode   400/15000  eps=0.980  retour_moyen=32.66  (12.1 ép/s)
  épisode   500/15000  eps=0.974  retour_moyen=32.45  (11.6 ép/s)
  épisode   600/15000  eps=0.969  retour_moyen=34.31  (11.6 ép/s)
  épisode   700/15000  eps=0.964  retour_moyen=32.81  (11.5 ép/s)
  épisode   800/15000  eps=0.958  retour_moyen=31.49  (11.5 ép/s)
  épisode   900/15000  eps=0.953  retour_moyen=32.98  (11.2 ép/s)
  épisode  1000/15000  eps=0.948  retour_moyen=32.18  (11.1 ép/s)
  épisode  1100/15000  eps=0.943  retour_moyen=33.18  (10.9 ép/s)
  épisode  1200/15000  eps=0.938  retour_moyen=35.31  (10.7 ép/s)
  épisode  1300/15000  eps=0.933  retour_moyen=34.40  (10.7 ép/s)
  épisode  1400/15000  eps=0.928  retour_moyen=33.96  (10.7 ép/s)
  épisode  1500/15000  eps=0.923  retour_moyen=34.67  (10.5 ép/s)
  épisode 

In [ ]:
# --- Courbe d'apprentissage + sauvegarde horodatée ----------------------
from registry import model_path, write_meta

w = 100
smooth = np.convolve(episode_returns, np.ones(w) / w, mode="valid")
plt.figure(figsize=(9, 4))
plt.plot(smooth)
plt.title("DQN — retour par épisode (moyenne glissante 100)")
plt.xlabel("Épisode"); plt.ylabel("Retour (récompense normalisée cumulée)")
plt.grid(alpha=0.3); plt.show()

final = model_path("dqn")                            # dqn_AAAAMMJJ-HHMMSS.keras
online.save(str(final))
write_meta(final, kind="dqn", episodes=len(episode_returns), horizon=HORIZON,
           steps_per_decision=env.steps_per_decision, seed=0, gamma=GAMMA, lr=LR,
           architecture=[l.units for l in online.layers if hasattr(l, "units")])
print(f"Modèle sauvegardé : {final.name}")
print("Les agents chargent cette version (la plus récente) par défaut : DQNAgent.load()")

NameError: name 'episode_returns' is not defined

: 

In [1]:
# resync la chaîne complète + les agents de pricing (au cas où le kernel a une version périmée)
_reload_project()                                   # refdata -> agents -> market -> gym_wrapper
for _name in ("registry", "agent_fixed", "agent_qtable", "agent_dqn", "evaluate"):
    importlib.reload(importlib.import_module(_name))
from gym_wrapper import TicketEnv

# --- Comparaison finale : baselines vs Q-table vs DQN ---------------------
from agent_fixed import FixedPriceAgent
from agent_qtable import QTableAgent
from agent_dqn import DQNAgent
from evaluate import compare_agents, summarize

agents = [
    FixedPriceAgent.from_price(70),        # prix fixe "réaliste" (remplir la salle)
    FixedPriceAgent.from_price(90),        # prix fixe optimum (inconnaissable a priori)
    QTableAgent.load(),                    # Q-table Monte-Carlo la plus récente
    DQNAgent(online),                     # ou DQNAgent.load() pour la dernière version sauvée
]
agents[0].name = "fixed"
agents[1].name = "fixed_90"

results = compare_agents(agents, lambda: TicketEnv(horizon=HORIZON), n_episodes=300)
summarize(results)

NameError: name '_reload_project' is not defined